# Seminar 2. Custom PyTorch Operators

# Building Models in PyTorch Through Composition

PyTorch models are built using **composition**.  
Instead of defining one large monolithic network, we construct models by combining smaller, reusable modules.

Each module can contain other modules, which allows us to build hierarchical and well-structured architectures.

---

## Composition

Composition means:

- A model is built from smaller blocks.
- Each block can contain multiple layers.
- Blocks can be reused in larger architectures.
- Complex models are created by stacking simpler components.

This keeps code:

- Modular  
- Reusable  
- Readable  
- Easy to extend  


## Key Ideas

- Inherit from `nn.Module`
- Define layers inside `__init__`
- Define computation in `forward()`
- Create reusable blocks
- Build larger models by combining blocks

---

## Example: Model Built from Two Blocks

Below is a simple example where:

- We define a reusable blocks: `LinearReLUBlock` and `LinearTanhBlock`
- The final model is composed of two such blocks


In [13]:
import torch
import torch.nn as nn
import math


class LinearReLUBlock(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features)
        self.activation = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = self.activation(x)
        return x


class LinearTanhBlock(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features)
        self.activation = nn.Tanh()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = self.activation(x)
        return x


class CombinedModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.block1 = LinearReLUBlock(4, 8)
        self.block2 = LinearTanhBlock(8, 8)
        self.output = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.block1(x)
        x = self.block2(x)
        x = self.output(x)
        return x


model = CombinedModel()
print(model)

CombinedModel(
  (block1): LinearReLUBlock(
    (linear): Linear(in_features=4, out_features=8, bias=True)
    (activation): ReLU()
  )
  (block2): LinearTanhBlock(
    (linear): Linear(in_features=8, out_features=8, bias=True)
    (activation): Tanh()
  )
  (output): Linear(in_features=8, out_features=2, bias=True)
)


# What `nn.Module` Enables

When we inherit from `nn.Module`, we automatically gain powerful functionality that works **recursively across all submodules**.

## What `nn.Module` Gives Us



### Parameter Registration

All layers assigned as attributes (e.g. `self.linear = nn.Linear(...)`) are:

- Automatically registered
- Collected in `model.parameters()`
- Included in `model.state_dict()`

This works **recursively** for all sub-blocks.

In [14]:
print("Registered parameters:")
for name, param in model.named_parameters():
    print(name, param.shape)


Registered parameters:
block1.linear.weight torch.Size([8, 4])
block1.linear.bias torch.Size([8])
block2.linear.weight torch.Size([8, 8])
block2.linear.bias torch.Size([8])
output.weight torch.Size([2, 8])
output.bias torch.Size([2])


### Automatic Gradient Tracking

During the forward pass:

- PyTorch dynamically builds a computation graph
- Calling `loss.backward()` computes gradients
- Gradients are stored in each parameter’s `.grad`

No manual graph management is required.

In [15]:
x = torch.randn(5, 4)
target = torch.randn(5, 2)

criterion = nn.MSELoss()
output = model(x)
loss = criterion(output, target)

loss.backward()

print("\nGradient computed for output layer:",
      model.output.weight.grad is not None)
model.output.weight.grad


Gradient computed for output layer: True


tensor([[ 0.0705,  0.0421, -0.0535, -0.0176,  0.0173, -0.0134, -0.0611, -0.0219],
        [ 0.0041, -0.0819,  0.1764, -0.0298, -0.2545,  0.0259,  0.1158, -0.0785]])

### Device and Type Transfer (`.to()`)

Calling:

    model.to(device)

or

    model.to(dtype)

moves all:

- Parameters
- Buffers
- Submodules

to CPU/GPU/dtype automatically.

In [16]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

dtype = torch.float32

model = model.to(device=device, dtype=dtype)

x: torch.Tensor = torch.randn(5, 4, device=device, dtype=dtype)

print("Model device:", next(model.parameters()).device)
print("Model dtype:", next(model.parameters()).dtype)

Model device: cuda:0
Model dtype: torch.float32


### Saving & Loading (`state_dict()`)

- `model.state_dict()` returns all parameters recursively
- `model.load_state_dict(...)` restores them

This works across the full module tree.

In [17]:
state_dict = model.state_dict()
torch.save(state_dict, "combined_model.pt")

# `train()` vs `eval()` Mode in PyTorch

PyTorch modules have two main modes: **training mode** and **evaluation mode**.  
Switching between them affects layers that behave differently during training and inference.

---

## `model.train()`

- Sets the model to **training mode**.
- Used when training the model with gradient updates.
- Affects certain layers, such as:

| Layer Type        | Behavior in `train()` Mode                  |
|------------------|--------------------------------------------|
| `Dropout`         | Randomly zeroes some activations           |
| `BatchNorm`       | Updates running statistics (mean/variance) |

- Gradients are computed as usual.

---

## `model.eval()`

- Sets the model to **evaluation (inference) mode**.
- Used when evaluating or deploying the model.
- Affects certain layers:

| Layer Type        | Behavior in `eval()` Mode                   |
|------------------|--------------------------------------------|
| `Dropout`         | Passes all activations through unchanged  |
| `BatchNorm`       | Uses stored running mean/variance         |

- No layers update internal statistics.
- Gradients are usually not required (often used with `torch.no_grad()`).

---

## Key Points

- Always use `model.train()` during training.
- Always use `model.eval()` during evaluation or testing.
- Forgetting to switch can lead to inconsistent results, especially with `Dropout` or `BatchNorm`.



In [18]:
import torch
import torch.nn as nn

# Simple model with Dropout and BatchNorm
class SimpleModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc1 = nn.Linear(4, 8)
        self.bn = nn.BatchNorm1d(8)
        self.dropout = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = self.bn(x)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x


model = SimpleModel()
x = torch.randn(5, 4)

# Training mode
model.train()
output_train = model(x)
print("Training mode output:\n", output_train)

# Evaluation mode
model.eval()
with torch.no_grad():
    output_eval = model(x)
print("Evaluation mode output:\n", output_eval)


Training mode output:
 tensor([[ 0.1701, -1.4929],
        [ 0.0025, -0.1735],
        [ 0.9414, -0.8225],
        [ 1.2773, -0.6540],
        [ 0.0750, -0.6011]], grad_fn=<AddmmBackward0>)
Evaluation mode output:
 tensor([[-0.0232, -0.4635],
        [-0.0084, -0.1837],
        [ 0.0827, -0.4371],
        [ 0.1685, -0.3241],
        [ 0.1306, -0.2423]])


# `torch.no_grad()` and `torch.inference_mode()` in PyTorch

When performing inference (evaluating a model without updating parameters), PyTorch provides context managers to **disable gradient tracking**. This saves memory and speeds up computation.

---

## `torch.no_grad()`

- Disables gradient tracking.
- Useful during evaluation or inference.
- Gradients are **not computed**, but autograd still tracks operations for some internal purposes.
- Can be used as a **context manager** or a **function decorator**.

---

## `torch.inference_mode()`

- Introduced in PyTorch 1.9.
- Similar to `no_grad()`, but **more efficient**.
- Completely disables autograd and reduces memory usage.
- Recommended for pure inference pipelines.



In [19]:
import torch
import torch.nn as nn

class SimpleModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc = nn.Linear(4, 2)

    # -----------------------------
    # Using torch.no_grad() as method decorator
    # -----------------------------
    @torch.no_grad()
    def forward_no_grad(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

    # -----------------------------
    # Using torch.inference_mode() as method decorator
    # -----------------------------
    @torch.inference_mode()
    def forward_inference(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

model = SimpleModel()
x = torch.randn(5, 4)

# -----------------------------
# Call decorated methods
# -----------------------------
output_no_grad_method = model.forward_no_grad(x)
output_infer_method = model.forward_inference(x)

print("Output no_grad method:\n", output_no_grad_method)
print("Output inference_mode method:\n", output_infer_method)

# -----------------------------
# Using context managers
# -----------------------------

with torch.no_grad():
    output_no_grad_cm = model(x)

with torch.inference_mode():
    output_infer_cm = model(x)

print("Output no_grad context manager:\n", output_no_grad_cm)
print("Output inference_mode context manager:\n", output_infer_cm)


Output no_grad method:
 tensor([[ 0.4050,  0.6532],
        [-0.1911,  1.2764],
        [-0.9465,  0.6733],
        [ 1.2627,  0.8396],
        [-0.0081,  0.2453]])
Output inference_mode method:
 tensor([[ 0.4050,  0.6532],
        [-0.1911,  1.2764],
        [-0.9465,  0.6733],
        [ 1.2627,  0.8396],
        [-0.0081,  0.2453]])
Output no_grad context manager:
 tensor([[ 0.4050,  0.6532],
        [-0.1911,  1.2764],
        [-0.9465,  0.6733],
        [ 1.2627,  0.8396],
        [-0.0081,  0.2453]])
Output inference_mode context manager:
 tensor([[ 0.4050,  0.6532],
        [-0.1911,  1.2764],
        [-0.9465,  0.6733],
        [ 1.2627,  0.8396],
        [-0.0081,  0.2453]])


# Disabling Gradients with `requires_grad_(False)`

PyTorch provides a convenient method `requires_grad_()` that can **enable or disable gradients in-place** for all parameters of a model or a tensor.

Using:

```python
param.requires_grad_(False)
```

- Sets `requires_grad=False` **in-place** for that parameter.
- This is useful for freezing models during inference or transfer learning.
- Can be applied to an entire model recursively by iterating over its parameters.


In [20]:
model = SimpleModel()

# Disable gradient computation for all parameters using requires_grad_()
for param in model.parameters():
    param.requires_grad_(False)

# Verify
for name, param in model.named_parameters():
    print(f"{name}: requires_grad={param.requires_grad}")

# Forward pass still works
x = torch.randn(5, 4)
output = model(x)
print("Output shape:", output.shape)

fc.weight: requires_grad=False
fc.bias: requires_grad=False
Output shape: torch.Size([5, 2])


# Redefining `train()` and `eval()` in `nn.Module`

PyTorch’s `nn.Module` provides built-in `train(mode: bool = True)` and `eval()` methods to switch between **training** and **evaluation** modes.  

Sometimes, when creating **custom modules or blocks**, you might want to **override these methods** to perform extra actions whenever the mode changes.

---

## Why Override?

- Apply mode-specific logic to sub-blocks or attributes that are not standard layers
- Log or track mode switches
- Automatically modify internal flags or buffers along with training/eval mode

---

## How It Works

- `train(mode: bool = True)` sets `self.training = mode` for the module
- `eval()` is equivalent to `train(False)`
- Default implementation recursively calls `train(mode)` on all submodules
- Overriding allows custom behavior while keeping recursive updates intact


In [21]:
import torch
import torch.nn as nn
from torch import Tensor
from typing import Self

class CustomBlock(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.linear = nn.Linear(4, 4)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = torch.relu(x)
        x = self.dropout(x)
        return x

    # -----------------------------
    # Override train() method
    # -----------------------------
    def train(self, mode: bool = True) -> Self:
        print(f"CustomBlock set to {'train' if mode else 'eval'} mode")
        super().train(mode)  # Call original method to update submodules
        # Add any custom logic here
        return self

    # -----------------------------
    # Override eval() method
    # -----------------------------
    def eval(self) -> Self:
        print("CustomBlock set to eval mode")
        return super().eval()


# Example usage
model = CustomBlock()
x = torch.randn(2, 4)

# Switch to training mode
model.train()
output_train = model(x)

# Switch to evaluation mode
model.eval()
with torch.no_grad():
    output_eval = model(x)

print("Output training mode:", output_train)
print("Output eval mode:", output_eval)


CustomBlock set to train mode
CustomBlock set to eval mode
CustomBlock set to eval mode
Output training mode: tensor([[0.0000, 0.0000, 0.2138, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000]], grad_fn=<MulBackward0>)
Output eval mode: tensor([[0.8760, 0.0000, 0.1069, 0.0000],
        [0.4499, 0.0000, 0.0000, 0.0000]])


# Common Module Aggregators in PyTorch

When building neural networks, it is often useful to group multiple layers or submodules together.  
PyTorch provides several **module aggregators** that help organize layers and blocks. The most common ones are:



## `nn.Sequential`

- Holds modules in a sequential order.
- Executes them **in the order they are added** during the forward pass.
- Ideal for simple **stacked layers** with a single input and output.

**Key points:**

- Forward pass is automatically defined.
- Cannot handle multiple inputs or branching.

In [22]:
seq_model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 2)
)

x = torch.randn(5, 4)
output_seq = seq_model(x)
print("nn.Sequential output shape:", output_seq.shape)


nn.Sequential output shape: torch.Size([5, 2])


## `nn.ModuleList`

- Holds a **list of modules**.
- Does **not define a forward pass automatically**.
- Useful when you need to **loop over modules**, or have conditional computation.

**Key points:**

- Modules are registered properly, so parameters are tracked.
- You must define your own `forward()`.

In [23]:
class ModuleListModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(4, 8),
            nn.ReLU(),
            nn.Linear(8, 2)
        ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x)
        return x

ml_model = ModuleListModel()
output_ml = ml_model(x)
print("nn.ModuleList output shape:", output_ml.shape)

nn.ModuleList output shape: torch.Size([5, 2])


## `nn.ModuleDict`

- Holds modules in a **dictionary** with string keys.
- Useful for architectures with **named branches**, **dynamic selection**, or **multi-head outputs**.
- Like `ModuleList`, it does **not define a forward pass**.

In [24]:
class ModuleDictModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.branches = nn.ModuleDict({
            "branch1": nn.Linear(4, 8),
            "branch2": nn.Linear(4, 8)
        })
        self.output: nn.Linear = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor, branch_name: str = "branch1") -> torch.Tensor:
        x = self.branches[branch_name](x)
        return self.output(x)

md_model = ModuleDictModel()
output_md = md_model(x, branch_name="branch2")
print("nn.ModuleDict output shape:", output_md.shape)

nn.ModuleDict output shape: torch.Size([5, 2])



## Работа на семинаре

### LSTM

![lstm](assets/LSTM.png)

In [25]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(LSTMCell, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        
        # (Input gate, Forget gate, Cell gate, Output gate)
        self.gates = nn.Linear(input_size + hidden_size, 4 * hidden_size)
    
    def forward(self, x, state):
        h_prev, c_prev = state
        
        combined = torch.cat([x, h_prev], dim=1)
        
        # Calculate gate activations
        gates_out = self.gates(combined)
        i_gate, f_gate, g_gate, o_gate = gates_out.chunk(4, dim=1)
        
        i_t = torch.sigmoid(i_gate)
        f_t = torch.sigmoid(f_gate)
        g_t = torch.tanh(g_gate)
        o_t = torch.sigmoid(o_gate)
        
        # Compute next cell state and next hidden state
        c_t = f_t * c_prev + i_t * g_t
        h_t = o_t * torch.tanh(c_t)
        
        return h_t, c_t


### Inception

![inception](assets/inception.png)

In [26]:
class InceptionModule(nn.Module):
    def __init__(self, in_channels, out_1x1,                    # branch 1
                                    yellow_3x3, out_3x3,        # branch 2
                                    yellow_1x1, out_5x5,        # branch 3  
                                    out_pool):                  # branch 4
        super(InceptionModule, self).__init__()
        
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, out_1x1, kernel_size=1),
            nn.BatchNorm2d(out_1x1),
            nn.ReLU(inplace=True)
        )
        
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, yellow_3x3, kernel_size=1),
            nn.BatchNorm2d(yellow_3x3),
            nn.ReLU(inplace=True),
            nn.Conv2d(yellow_3x3, out_3x3, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_3x3),
            nn.ReLU(inplace=True)
        )
        
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, yellow_1x1, kernel_size=1),
            nn.BatchNorm2d(yellow_1x1),
            nn.ReLU(inplace=True),
            nn.Conv2d(yellow_1x1, out_5x5, kernel_size=5, padding=2),
            nn.BatchNorm2d(out_5x5),
            nn.ReLU(inplace=True)
        )
        
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            nn.Conv2d(in_channels, out_pool, kernel_size=1),
            nn.BatchNorm2d(out_pool),
            nn.ReLU(inplace=True)
        )
        
    def forward(self, x):
        return torch.cat([self.branch1(x), 
                          self.branch2(x), 
                          self.branch3(x), 
                          self.branch4(x)], dim=1)

model = InceptionModule(3, 4, 4, 4, 4, 4, 4)
input = torch.randn((1, 3, 224, 224))

model(input).shape


torch.Size([1, 16, 224, 224])

### SE

![se](assets/SqueezeAndExcite.png)

In [27]:
class SEBlock(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super(SEBlock, self).__init__()
        
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        
        self.excite = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // reduction, in_channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        batch, channels, _, _ = x.size()
        
        # Squeeze
        y = self.squeeze(x).view(batch, channels)
        
        # Excite
        y = self.excite(y).view(batch, channels, 1, 1)
        
        return x * y

model = SEBlock(3)
input = torch.randn((1, 3, 224, 224))
model(input).shape


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/linear.py:124: UserWarning: Initializing zero-element tensors is a no-op
  init.kaiming_uniform_(self.weight, a=math.sqrt(5))


torch.Size([1, 3, 224, 224])

### Selective Kernel

![selective](assets/SelectiveKernel.png)


Не совсем понял по картинке, поискал и разобрался как устроена эта модель. Не вижу смысла копипастить код. Понятный и легко читаемый код в pytorch https://github.com/developer0hye/SKNet-PyTorch/blob/master/sknet.py (оригинальная статья использует другой фреймворк)

### PatchMerger

![patchmerger](assets/PatchMerger.png)


In [28]:
class PatchMerger(nn.Module):
    def __init__(self, dim):
        super(PatchMerger, self).__init__()
        self.norm = nn.LayerNorm(4 * dim)
        self.reduction = nn.Linear(4 * dim, 2 * dim, bias=False)

    def forward(self, x):
        # x is expected to be shape: (Batch, Height, Width, Channels)
        B, H, W, C = x.shape
        
        assert H % 2 == 0 and W % 2 == 0, "Spatial dimensions must be an even number."
        
        # Slice into 2x2 local regions
        x0 = x[:, 0::2, 0::2, :]  # Top-Left
        x1 = x[:, 1::2, 0::2, :]  # Bottom-Left
        x2 = x[:, 0::2, 1::2, :]  # Top-Right
        x3 = x[:, 1::2, 1::2, :]  # Bottom-Right

        # Concatenate 4 patches into one
        x_merged = torch.cat([x0, x1, x2, x3], dim=-1) # (B, H/2, W/2, 4C)
        x_merged = x_merged.view(B, (H // 2) * (W // 2), 4 * C) # Flatten spatially
        
        # Map 4C back down to 2C
        x_merged = self.norm(x_merged)
        x_reduced = self.reduction(x_merged) 
        
        # Reshape to expected feature map (B, H/2, W/2, 2C)
        out = x_reduced.view(B, H // 2, W // 2, 2 * C)
        return out


## Homework

2 задания:
1. Реализуйте требуемый в заголовке блок (максмсум 0.8 балов).

## ResNet Block (0.1 балл)

![Resnet](assets/ResBlock.png)

https://arxiv.org/pdf/1512.03385

In [29]:
class ResidualBlock(nn.Module):
    def __init__(self, block):
        super().__init__()
        self.block = block

    def forward(self, x):
        
        return x + self.block(x)

## Depthwise Separable Convolution (0.1 балл)
![DepthWiseConv](assets/DepthWiseConv.png)

https://arxiv.org/pdf/1610.02357

In [30]:
class SeparableConv2d(nn.Module):
    def __init__(self, in_size, out_size, kernel_size=3, stride=1, padding=1):
        super().__init__()

        self.depth_wise = nn.Sequential(
            nn.Conv2d(in_size, in_size, kernel_size=kernel_size, 
                      stride=stride, padding=padding, groups=in_size, bias=False),
            nn.BatchNorm2d(in_size),
        )

        self.conv_1x1 = nn.Conv2d(in_size, out_size, kernel_size=1, stride=1, padding=0, bias=False)


    def forward(self, x):
        
        return self.conv_1x1(self.depth_wise(x))


B, C_in, C_out = 2, 16, 32  
H, W = 64, 64               
kernel = 3

model = SeparableConv2d(in_size=C_in, out_size=C_out, kernel_size=kernel)
x = torch.randn(B, C_in, H, W)

out = model(x)
print(f"SeparableConv2d Output Shape: {out.shape}")

SeparableConv2d Output Shape: torch.Size([2, 32, 64, 64])


## Vanilla Attention (0.1 балл)

Let:

$$
\text{query} \in \mathbb{R}^{B \times d} \\
\text{key} \in \mathbb{R}^{B \times L \times d}
$$

---

### Alignment Scores

$$
\text{score} = \text{key} \cdot (W_\text{align} \, \text{query})^T \\
\text{score} \in \mathbb{R}^{B \times L}
$$

---

### Attention Weights

$$
\text{att} = \text{softmax}(\text{score}, \text{dim}=1) \\
\text{att} \in \mathbb{R}^{B \times L}
$$

---

### Context Vector

$$
\text{context} = \sum_{i=1}^{L} \text{att}_i \cdot \text{key}_i \\
\text{context} \in \mathbb{R}^{B \times d}
$$

---

### Output

$$
\text{out} = \tanh(W_\text{value} \, \text{context} + W_\text{query} \, \text{query}) \\
\text{out} \in \mathbb{R}^{B \times d}
$$



https://arxiv.org/abs/1409.0473


https://arxiv.org/abs/1508.04025

In [31]:
from typing import Optional
import torch
from torch import nn
import numpy as np

class VanillaAttention(nn.Module):
    def __init__(self, d: int):
        super().__init__()
        self.W_align = nn.Linear(d, d, bias=False)
        self.W_value = nn.Linear(d, d, bias=False)
        self.W_query = nn.Linear(d, d, bias=False)


    def forward(self, query: torch.Tensor, key: torch.Tensor):
        """
        query: (B, d)
        key:   (B, L, d)
        """
        #key @ (W_align @ query)^T  ->  (B, L)
        projected_query = self.W_align(query)                    # (B, d)
        score = torch.bmm(key, projected_query.unsqueeze(2))     # (B, L, 1)
        score = score.squeeze(2)                                 # (B, L)
        
        att = torch.softmax(score, dim=1)                        # (B, L)
        context = torch.bmm(att.unsqueeze(1), key).squeeze(1)    # (B, d)
        
        out = torch.tanh(self.W_value(context) + self.W_query(query))  # (B, d)
        return out

B, L, d = 2, 5, 8
model = VanillaAttention(d)
q = torch.randn(B, d)
k = torch.randn(B, L, d)
out = model(q, k)
print(f"VanillaAttention Output Shape: {out.shape}")

VanillaAttention Output Shape: torch.Size([2, 8])


## Dot Product Attention (0.1 балл)

$$
Q \in \mathbb{R}^{B \times L_q \times d_k} \\
K \in \mathbb{R}^{B \times L_k \times d_k} \\
V \in \mathbb{R}^{B \times L_k \times d_k}
$$

$$
S = \frac{Q K^T}{\sqrt{d_k}}
$$

$$
\text{Attention}(Q, K, V) = \text{softmax}(S, \text{dim}=-1) \, V
$$



https://arxiv.org/abs/1706.03762


In [32]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self):
        super().__init__()


    def forward(self, Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor,mask: torch.Tensor = None):
        """
        Q: (B, L_q, d_k)
        K: (B, L_k, d_k)
        V: (B, L_k, d_k)
        mask: (B, L_q, L_k) 
        """
        d_k = Q.size(-1)
        scores = torch.matmul(Q, K.transpose(1, 2)) / math.sqrt(d_k)  # (B, L_q, L_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = torch.softmax(scores, dim=-1)                        # (B, L_q, L_k)
        output = torch.bmm(attn, V)                                 # (B, L_q, d_k)
        return output


B, L_q, L_k, d_k = 2, 4, 6, 8
model = ScaledDotProductAttention()
Q = torch.randn(B, L_q, d_k)
K = torch.randn(B, L_k, d_k)
V = torch.randn(B, L_k, d_k)
mask = torch.ones(B, L_q, L_k) 
out = model(Q, K, V, mask=mask)
print(f"ScaledDotProductAttention Output Shape: {out.shape}") 

ScaledDotProductAttention Output Shape: torch.Size([2, 4, 8])


## Multihead Attention (0.1 балл)

![MultiheadAttention](assets/MultiheadAttention.webp)

https://arxiv.org/abs/1706.03762


In [33]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        
        assert d_model % num_heads == 0
        self.d_k = d_model // num_heads
        self.num_heads = num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor, mask: torch.Tensor = None):
        """
        Q: (B, L_q, d_model)
        K: (B, L_k, d_model)
        V: (B, L_k, d_model)
        """
        B = Q.size(0)

        Q = self.W_q(Q).view(B, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(K).view(B, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(V).view(B, -1, self.num_heads, self.d_k).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn = torch.softmax(scores, dim=-1)              # (B, h, L_q, L_k)
        context = torch.matmul(attn, V)                   # (B, h, L_q, d_k)

        context = context.transpose(1, 2).contiguous().view(B, -1, self.num_heads * self.d_k)

        return self.W_o(context)

B, L_q, L_k, d_model = 2, 4, 6, 16
num_heads = 4
model = MultiHeadAttention(d_model, num_heads)
Q = torch.randn(B, L_q, d_model)
K = torch.randn(B, L_k, d_model)
V = torch.randn(B, L_k, d_model)
out = model(Q, K, V)
print(f"MultiHeadAttention Output Shape: {out.shape}")

MultiHeadAttention Output Shape: torch.Size([2, 4, 16])


## Transformer Encoder Layer (0.1 балл)


![Transformer Encoder Layer](assets/TransformerEncoder.png)


https://arxiv.org/abs/1706.03762

In [34]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int = 2048, dropout: float = 0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn  = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )
        
    def forward(self, x: torch.Tensor, mask: torch.Tensor = None):
        """x: (B, L, d_model)"""

        x_norm = self.norm1(x)
        attn_out, _ = self.attn(x_norm, x_norm, x_norm, attn_mask=mask)
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        
        return x


B, L, d_model = 2, 10, 32
num_heads = 4
model = TransformerEncoderLayer(d_model=d_model, num_heads=num_heads, d_ff=128)
x = torch.randn(B, L, d_model)
out = model(x)
print(f"TransformerEncoderLayer Output Shape: {out.shape}")

TransformerEncoderLayer Output Shape: torch.Size([2, 10, 32])


## MLP Mixer (0.1 балл)


![MLPMixer](assets/MLPMixer.png)


https://arxiv.org/abs/2105.01601

In [35]:
class MLPMixerBlock(nn.Module):
    def __init__(self, num_patches: int, d_model: int, token_mlp_dim: int, channel_mlp_dim: int):
        super().__init__()
        
        self.norm1 = nn.LayerNorm(d_model)
        self.token_mixing = nn.Sequential(
            nn.Linear(num_patches, token_mlp_dim),
            nn.GELU(),
            nn.Linear(token_mlp_dim, num_patches),
        )
        self.norm2 = nn.LayerNorm(d_model)
        self.channel_mixing = nn.Sequential(
            nn.Linear(d_model, channel_mlp_dim),
            nn.GELU(),
            nn.Linear(channel_mlp_dim, d_model),
        )
    
    def forward(self, x: torch.Tensor):
        """x: (B, num_patches, d_model)"""
        
        # Token-mixing: MLP across patches (transpose → MLP → transpose)
        y = self.norm1(x)                # (B, S, C)
        y = y.transpose(1, 2)           # (B, C, S)
        y = self.token_mixing(y)        # (B, C, S)
        y = y.transpose(1, 2)           # (B, S, C)
        x = x + y
        
        # Channel-mixing: MLP across channels
        y = self.norm2(x)
        y = self.channel_mixing(y)
        x = x + y

        return x


B, num_patches, d_model = 2, 64, 128
model = MLPMixerBlock(num_patches=num_patches, d_model=d_model, token_mlp_dim=256, channel_mlp_dim=512)
x = torch.randn(B, num_patches, d_model)
out = model(x)
print(f"MLPMixerBlock Output Shape: {out.shape}")

MLPMixerBlock Output Shape: torch.Size([2, 64, 128])


## ConvMixer (0.1 балл)

![ConvMixer](assets/ConvMixer.png)


https://arxiv.org/abs/2201.09792

In [36]:
class ConvMixer(nn.Module):
    def __init__(self, dim: int, kernel_size: int = 9):
        super().__init__()
    
        self.depthwise = nn.Sequential(
            nn.Conv2d(dim, dim, kernel_size, padding=kernel_size // 2, groups=dim),
            nn.ReLU(),
            nn.BatchNorm2d(dim),
        )
    
        self.pointwise = nn.Sequential(
            nn.Conv2d(dim, dim, kernel_size=1),
            nn.ReLU(),
            nn.BatchNorm2d(dim),
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, C, H, W)"""

        x = x + self.depthwise(x)
        x = self.pointwise(x)

        return x


B, C, H, W = 2, 64, 32, 32
model = ConvMixer(dim=C, kernel_size=9)
x = torch.randn(B, C, H, W)
out = model(x)
print(f"ConvMixer Output Shape: {out.shape}")

ConvMixer Output Shape: torch.Size([2, 64, 32, 32])


## Вопрос (0.2 балла)

Объясните, почему MLPMixer, ConvMixer может работать почти так же эффективно, как обычный Multihead Attention.

Напишите формулу, связывающую Multihead Attention, ConvMixer и MLPMixer

Опишите преимущества и недостатки между ConvMixer, MLPMixer и Multihead Attention

---

Ответ:

Все три архитектуры выполняют одну и ту же концептуальную задачу: смешивание информации между токенами (пространственное/последовательное) и 
смешивание информации между каналами (признаками). Разница лишь в том, как реализовано межтокенное смешивание

- Attention — через взвешенное суммирование на основе сходства
- MLPMixer — через линейные слои (token-mixing и channel-mixing)
- ConvMixer — через depthwise (пространственное смешивание) и pointwise (смешивание каналов)

Любой слой, объединяющий информацию между токенами, можно представить как

$\text{Output} = A \cdot V$ 
 где матрица смешивания $A$ определяется архитектурой:
- MHA: $A = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)$.Свойство: Data-dependent (динамическая). Веса вычисляются «на лету» на основе схожести входных данных.
- MLPMixer: $A = W$.Свойство: Static, Global. Матрица весов обучается один раз и фиксирована при инференсе. Смешивает все токены со всеми.
- ConvMixer: $A = \text{Sparse } W$ (Depthwise Convolution).Свойство: Static, Local. Фиксированная разреженная матрица с локальной структурой (определяется размером ядра свертки).
Разница между этими SOTA-архитектурами сводится к тому, является ли матрица смешивания $w_{ij}$ динамической (зависит от входа) или статической (выученной), а также к степени её разреженности (локальности).

Преимущества и недостатки

Multihead Attention даёт сильный индуктивный bias на длинных зависимостях и позволяет анализировать веса attention, но имеет квадратичную по длине последовательности сложность по памяти и вычислениям $O(L^2)$.

MLPMixer работает за $O(L)$ по длине, проще в реализации и не использует позиционное кодирование. Однако у него фиксированный receptive field, и на очень длинных последовательностях он может проигрывать attention.

ConvMixer имеет сложность $O(1)$ по длине последовательности, использует локальную связность и хорошо масштабируется. Минусы: ограниченный локальный receptive field, который расширяется только при увеличении размера kernel.